# 702 — Compound Prioritization

## Objective

Reconcile chemical redundancy and organize the frozen Notebook 701 perturbational hypotheses without altering perturbagen-level primary inference.

Notebook 702 consumes the registered Notebook 701 handoffs and the frozen Notebook 110 compound annotation. It preserves every Notebook 701 program × pert_id result, assigns exact-structure identity under the prospectively frozen hierarchy, quantifies chemical redundancy and within-structure heterogeneity, and emits a complete perturbational-hypothesis catalog.

## Scientific boundary

The analytical unit of primary inference remains the LINCS pert_id from Notebook 701. Exact-structure consolidation is descriptive and cannot create, rescue, or redefine primary support.

This notebook therefore:

1. preserves the Notebook 701 primary support state for every program × pert_id hypothesis;
2. reconciles exact chemical identity using full InChIKey first and canonical SMILES only under the frozen fallback rules;
3. retains ambiguous or unavailable chemical identity explicitly rather than forcing merges;
4. summarizes multiple pert_id representations of one exact structure without selecting a favorable representation;
5. carries forward lineage breadth/heterogeneity, prespecified sensitivity results, and perturbational-context diagnostics as transparent context;
6. emits the complete hypothesis catalog rather than only selected hits.

Notebook 702 does not perform target or mechanism-of-action aggregation; that is reserved for Notebook 703. It does not use Phase 5 or Phase 6 evidence, does not create an opaque composite score, and does not establish therapeutic efficacy, therapeutic reversal, causal mechanism, or validated targets.

## Stable outputs

The frozen Phase 7 contract defines four Notebook 702 handoffs:

- phase7.702.perturbagen_structure_map
- phase7.702.structure_connectivity_summary
- phase7.702.perturbational_hypothesis_catalog
- phase7.702.analysis_metadata


In [1]:
# =============================================================================
# Imports and frozen Notebook 702 constants
# =============================================================================

import json
import platform
from importlib.metadata import version as package_version

import numpy as np
import pandas as pd

from pancancer_epigenetics.utils.artifact_registry import (
    load_artifact_registry,
    resolve_artifact_path,
    validate_artifact_registry,
)
from pancancer_epigenetics.utils.file_checks import (
    calculate_sha256,
)
from pancancer_epigenetics.utils.paths import (
    Paths,
    project_relative_path,
)

PROGRAM_IDS = (
    "CONSENSUS_TX_01",
    "CONSENSUS_TX_02",
    "CONSENSUS_TX_03",
)

PRIMARY_FAMILY_SIZE = 23742
PRIMARY_PERTURBAGEN_COUNT = 7914

PRIMARY_STATUS_ORDER = {
    "PRIMARY_BY_SUPPORTED": 0,
    "INVERSE_NOT_BY_SUPPORTED": 1,
    "NON_INVERSE": 2,
    "NOT_EVALUABLE": 3,
}

SENSITIVITY_TYPES = {
    "landmark_weighted_cosine": "landmark",
    "pooled_cell_line_weighted_cosine": "pooled_cell_line",
    "is_hiq_weighted_cosine": "is_hiq",
    "cmap_style_raw_wtcs": "cmap_style",
}

DIAGNOSTIC_COLUMNS = {
    "tas": "diagnostic_tas",
    "ss_ngene": "diagnostic_ss_ngene",
    "cc_q75": "diagnostic_cc_q75",
    "HALLMARK_E2F_TARGETS": "diagnostic_hallmark_e2f_targets",
    "HALLMARK_G2M_CHECKPOINT": "diagnostic_hallmark_g2m_checkpoint",
    "HALLMARK_P53_PATHWAY": "diagnostic_hallmark_p53_pathway",
    "HALLMARK_APOPTOSIS": "diagnostic_hallmark_apoptosis",
    "HALLMARK_UNFOLDED_PROTEIN_RESPONSE": (
        "diagnostic_hallmark_unfolded_protein_response"
    ),
}

print("Notebook 702 specification: FROZEN")
print("Programs:", PROGRAM_IDS)
print("Primary hypotheses:", f"{PRIMARY_FAMILY_SIZE:,}")
print("Primary perturbagens:", f"{PRIMARY_PERTURBAGEN_COUNT:,}")


Notebook 702 specification: FROZEN
Programs: ('CONSENSUS_TX_01', 'CONSENSUS_TX_02', 'CONSENSUS_TX_03')
Primary hypotheses: 23,742
Primary perturbagens: 7,914


In [2]:
# =============================================================================
# Resolve frozen Notebook 701 and compound-annotation inputs
# =============================================================================

artifact_registry = load_artifact_registry()

INPUT_ARTIFACT_IDS = (
    "phase7.701.primary_perturbagen_connectivity",
    "phase7.701.sensitivity_connectivity",
    "phase7.701.perturbational_context_diagnostics",
    "phase7.701.analysis_metadata",
    "phase1.110.primary_compound_annotation",
)

missing_input_ids = [
    artifact_id
    for artifact_id in INPUT_ARTIFACT_IDS
    if artifact_id not in artifact_registry["artifacts"]
]

if missing_input_ids:
    raise RuntimeError(
        "Notebook 702 requires frozen Notebook 701 and Notebook 110 "
        "handoffs. Missing artifacts: "
        + ", ".join(missing_input_ids)
    )

input_paths = {
    artifact_id: resolve_artifact_path(
        artifact_registry,
        artifact_id,
    )
    for artifact_id in INPUT_ARTIFACT_IDS
}

input_hashes = {
    artifact_id: artifact_registry[
        "artifacts"
    ][artifact_id]["sha256"]
    for artifact_id in INPUT_ARTIFACT_IDS
}


In [3]:
# =============================================================================
# Load frozen Notebook 701 and compound-annotation handoffs
# =============================================================================

primary_connectivity = pd.read_csv(
    input_paths[
        "phase7.701.primary_perturbagen_connectivity"
    ],
    low_memory=False,
)

sensitivity_connectivity = pd.read_parquet(
    input_paths[
        "phase7.701.sensitivity_connectivity"
    ]
)

context_diagnostics = pd.read_parquet(
    input_paths[
        "phase7.701.perturbational_context_diagnostics"
    ]
)

with input_paths[
    "phase7.701.analysis_metadata"
].open("r", encoding="utf-8") as handle:
    notebook701_metadata = json.load(handle)

compound_annotation = pd.read_csv(
    input_paths[
        "phase1.110.primary_compound_annotation"
    ],
    low_memory=False,
)


In [4]:
# =============================================================================
# Validate the local Notebook 702 input gate
# =============================================================================

contract_path = (
    Paths.root
    / "docs"
    / "contracts"
    / "phase7"
    / "PHASE7_ANALYSIS_CONTRACT.md"
)

contract_sha256 = calculate_sha256(
    contract_path
)

if (
    notebook701_metadata["contract"]["sha256"]
    != contract_sha256
):
    raise RuntimeError(
        "Notebook 701 and Notebook 702 do not reference the same "
        "frozen Phase 7 contract."
    )

if len(primary_connectivity) != PRIMARY_FAMILY_SIZE:
    raise RuntimeError(
        "Unexpected Notebook 701 primary hypothesis count."
    )

if (
    primary_connectivity["pert_id"].nunique()
    != PRIMARY_PERTURBAGEN_COUNT
):
    raise RuntimeError(
        "Unexpected Notebook 701 perturbagen universe."
    )

if primary_connectivity.duplicated(
    ["program_id", "pert_id"]
).any():
    raise RuntimeError(
        "Notebook 701 primary hypothesis keys are not unique."
    )

if not set(
    primary_connectivity["primary_support_status"].unique()
).issubset(PRIMARY_STATUS_ORDER):
    raise RuntimeError(
        "Unexpected Notebook 701 primary support state."
    )

primary_perturbagen_ids = set(
    primary_connectivity["pert_id"]
)

annotation_perturbagen_ids = set(
    compound_annotation["pert_id"]
)

if primary_perturbagen_ids != annotation_perturbagen_ids:
    raise RuntimeError(
        "Frozen compound annotation does not exactly cover the "
        "Notebook 701 perturbagen universe."
    )

print("NOTEBOOK 702 INPUT GATE: PASS")
print("Registered inputs:", len(INPUT_ARTIFACT_IDS))
print("Primary hypotheses:", f"{len(primary_connectivity):,}")
print("Annotated perturbagens:", f"{len(annotation_perturbagen_ids):,}")


NOTEBOOK 702 INPUT GATE: PASS
Registered inputs: 5
Primary hypotheses: 23,742
Annotated perturbagens: 7,914


## Exact-structure reconciliation

Chemical identity is reconciled only after perturbagen-level inference has been frozen.

Full InChIKey is authoritative when available. Canonical SMILES is used only as the prospectively specified fallback. A SMILES-only perturbagen can join an existing full-InChIKey group only when that SMILES maps unambiguously to exactly one full InChIKey in the frozen primary annotation universe.

The provider literal restricted is unavailable identity information. Ambiguous SMILES-only entities and perturbagens lacking usable structure metadata remain perturbagen-specific and are not collapsed. cmap_name and compound aliases remain descriptive metadata only.


In [5]:
# =============================================================================
# Define chemical-identity normalization and serialization helpers
# =============================================================================

def normalize_structure_value(value):
    """Return a trimmed usable structure identifier or pandas NA."""
    if pd.isna(value):
        return pd.NA

    text = str(value).strip()

    if (
        not text
        or text.casefold() == "restricted"
    ):
        return pd.NA

    return text


def json_unique_values(values):
    """Serialize non-missing unique values in deterministic order."""
    unique_values = sorted(
        {
            str(value)
            for value in values
            if pd.notna(value)
            and str(value) != ""
        }
    )

    return json.dumps(
        unique_values,
        ensure_ascii=False,
        separators=(",", ":"),
    )


def merge_json_value_lists(values):
    """Merge JSON-list cells without using names as analytical identity."""
    merged = set()

    for value in values:
        if pd.isna(value):
            continue

        merged.update(
            json.loads(value)
        )

    return json.dumps(
        sorted(merged),
        ensure_ascii=False,
        separators=(",", ":"),
    )


def first_nonmissing(values):
    """Return the sole non-missing value after uniqueness validation."""
    observed = [
        value
        for value in values
        if pd.notna(value)
    ]

    return (
        observed[0]
        if observed
        else pd.NA
    )


In [6]:
# =============================================================================
# Materialize perturbagen-level chemical identity candidates
# =============================================================================

identity_rows = (
    compound_annotation[
        [
            "pert_id",
            "cmap_name",
            "canonical_smiles",
            "inchi_key",
            "compound_aliases",
        ]
    ]
    .copy()
)

identity_rows[
    "usable_full_inchi_key"
] = identity_rows[
    "inchi_key"
].map(
    normalize_structure_value
)

identity_rows[
    "usable_canonical_smiles"
] = identity_rows[
    "canonical_smiles"
].map(
    normalize_structure_value
)

identity_multiplicity = (
    identity_rows
    .groupby(
        "pert_id",
        as_index=False,
        sort=True,
    )
    .agg(
        n_compoundinfo_rows=(
            "pert_id",
            "size",
        ),
        n_full_inchi_keys=(
            "usable_full_inchi_key",
            "nunique",
        ),
        n_canonical_smiles=(
            "usable_canonical_smiles",
            "nunique",
        ),
    )
)


In [7]:
# =============================================================================
# Validate within-perturbagen structure identity
# =============================================================================

multiple_full_inchi = (
    identity_multiplicity.loc[
        identity_multiplicity[
            "n_full_inchi_keys"
        ].gt(1),
        "pert_id",
    ]
    .tolist()
)

if multiple_full_inchi:
    raise RuntimeError(
        "Frozen pert_id maps to multiple usable full InChIKeys: "
        + ", ".join(multiple_full_inchi[:20])
    )

# The frozen Notebook 110 audit observed no multi-SMILES perturbagen.
# Halting here avoids inventing a post hoc rule if that upstream property changes.
multiple_smiles = (
    identity_multiplicity.loc[
        identity_multiplicity[
            "n_canonical_smiles"
        ].gt(1),
        "pert_id",
    ]
    .tolist()
)

if multiple_smiles:
    raise RuntimeError(
        "Frozen pert_id maps to multiple usable canonical SMILES. "
        "Notebook 702 has no frozen rule for choosing among them: "
        + ", ".join(multiple_smiles[:20])
    )

usable_inchi_lengths = (
    identity_rows[
        "usable_full_inchi_key"
    ]
    .dropna()
    .astype(str)
    .str.len()
)

if (
    not usable_inchi_lengths.empty
    and usable_inchi_lengths.lt(14).any()
):
    raise RuntimeError(
        "At least one usable InChIKey is too short to expose the "
        "14-character connectivity block."
    )

print("WITHIN-PERTURBAGEN CHEMICAL IDENTITY: VALIDATED")
print(
    "Perturbagens with full InChIKey:",
    f"{identity_multiplicity['n_full_inchi_keys'].eq(1).sum():,}",
)
print(
    "Perturbagens with canonical SMILES:",
    f"{identity_multiplicity['n_canonical_smiles'].eq(1).sum():,}",
)


WITHIN-PERTURBAGEN CHEMICAL IDENTITY: VALIDATED
Perturbagens with full InChIKey: 7,389
Perturbagens with canonical SMILES: 7,603


In [8]:
# =============================================================================
# Collapse descriptive compound metadata to one row per perturbagen
# =============================================================================

perturbagen_identity = (
    identity_rows
    .groupby(
        "pert_id",
        as_index=False,
        sort=True,
    )
    .agg(
        cmap_names_json=(
            "cmap_name",
            json_unique_values,
        ),
        compound_aliases_json=(
            "compound_aliases",
            json_unique_values,
        ),
        full_inchi_key=(
            "usable_full_inchi_key",
            first_nonmissing,
        ),
        canonical_smiles=(
            "usable_canonical_smiles",
            first_nonmissing,
        ),
    )
    .merge(
        identity_multiplicity[
            [
                "pert_id",
                "n_compoundinfo_rows",
            ]
        ],
        on="pert_id",
        how="left",
        validate="one_to_one",
    )
)

if len(perturbagen_identity) != PRIMARY_PERTURBAGEN_COUNT:
    raise RuntimeError(
        "Perturbagen-level identity table lost frozen perturbagens."
    )


In [9]:
# =============================================================================
# Build frozen canonical-SMILES to full-InChIKey relationships
# =============================================================================

smiles_inchi_pairs = (
    perturbagen_identity.loc[
        perturbagen_identity[
            "canonical_smiles"
        ].notna()
        & perturbagen_identity[
            "full_inchi_key"
        ].notna(),
        [
            "canonical_smiles",
            "full_inchi_key",
        ],
    ]
    .drop_duplicates()
)

smiles_to_inchi = (
    smiles_inchi_pairs
    .groupby(
        "canonical_smiles",
        sort=True,
    )[
        "full_inchi_key"
    ]
    .agg(
        lambda values: tuple(
            sorted(
                set(
                    values.astype(str)
                )
            )
        )
    )
    .to_dict()
)

ambiguous_smiles = {
    smiles
    for smiles, inchi_keys in (
        smiles_to_inchi.items()
    )
    if len(inchi_keys) > 1
}

print(
    "Canonical SMILES linked to >1 full InChIKey:",
    f"{len(ambiguous_smiles):,}",
)


Canonical SMILES linked to >1 full InChIKey: 8


In [10]:
# =============================================================================
# Define exact-structure assignment under the frozen hierarchy
# =============================================================================

def assign_exact_structure_identity(
    pert_id,
    full_inchi_key,
    canonical_smiles,
):
    """Apply the frozen full-InChIKey-first reconciliation hierarchy."""
    matching_inchi_keys = (
        smiles_to_inchi.get(
            canonical_smiles,
            tuple(),
        )
        if pd.notna(canonical_smiles)
        else tuple()
    )

    # Full InChIKey remains authoritative even when its SMILES is shared
    # across incompatible full InChIKeys elsewhere in the frozen universe.
    if pd.notna(full_inchi_key):
        return {
            "resolved_full_inchi_key": str(
                full_inchi_key
            ),
            "exact_structure_id": (
                "INCHIKEY::"
                + str(full_inchi_key)
            ),
            "structure_identity_basis": (
                "FULL_INCHIKEY"
            ),
            "smiles_inchi_conflict": (
                pd.notna(canonical_smiles)
                and len(matching_inchi_keys) > 1
            ),
        }

    if pd.notna(canonical_smiles):
        if len(matching_inchi_keys) == 1:
            resolved_full_inchi_key = (
                matching_inchi_keys[0]
            )

            return {
                "resolved_full_inchi_key": (
                    resolved_full_inchi_key
                ),
                "exact_structure_id": (
                    "INCHIKEY::"
                    + resolved_full_inchi_key
                ),
                "structure_identity_basis": (
                    "SMILES_TO_UNIQUE_INCHIKEY"
                ),
                "smiles_inchi_conflict": False,
            }

        if len(matching_inchi_keys) == 0:
            return {
                "resolved_full_inchi_key": pd.NA,
                "exact_structure_id": (
                    "SMILES::"
                    + str(canonical_smiles)
                ),
                "structure_identity_basis": (
                    "CANONICAL_SMILES_FALLBACK"
                ),
                "smiles_inchi_conflict": False,
            }

        # Ambiguous SMILES-only entities remain perturbagen-specific;
        # forcing a merge would violate the frozen reconciliation rule.
        return {
            "resolved_full_inchi_key": pd.NA,
            "exact_structure_id": (
                "UNRESOLVED_SMILES::"
                + pert_id
            ),
            "structure_identity_basis": (
                "SMILES_AMBIGUOUS_UNRESOLVED"
            ),
            "smiles_inchi_conflict": True,
        }

    return {
        "resolved_full_inchi_key": pd.NA,
        "exact_structure_id": (
            "UNKNOWN_PERT_ID::"
            + pert_id
        ),
        "structure_identity_basis": (
            "UNKNOWN_PERT_ID"
        ),
        "smiles_inchi_conflict": False,
    }


In [11]:
# =============================================================================
# Materialize exact-structure assignments
# =============================================================================

structure_assignment_rows = []

for row in perturbagen_identity.itertuples(
    index=False
):
    structure_assignment_rows.append(
        {
            "pert_id": row.pert_id,
            **assign_exact_structure_identity(
                pert_id=row.pert_id,
                full_inchi_key=(
                    row.full_inchi_key
                ),
                canonical_smiles=(
                    row.canonical_smiles
                ),
            ),
        }
    )

structure_assignments = pd.DataFrame(
    structure_assignment_rows
)

perturbagen_structure_map = (
    perturbagen_identity
    .merge(
        structure_assignments,
        on="pert_id",
        how="left",
        validate="one_to_one",
    )
)


In [12]:
# =============================================================================
# Add InChIKey connectivity-block diagnostics
# =============================================================================

perturbagen_structure_map[
    "inchi_connectivity_block"
] = (
    perturbagen_structure_map[
        "resolved_full_inchi_key"
    ]
    .astype("string")
    .str.slice(0, 14)
)

full_inchi_by_block = (
    perturbagen_structure_map.loc[
        perturbagen_structure_map[
            "resolved_full_inchi_key"
        ].notna(),
        [
            "inchi_connectivity_block",
            "resolved_full_inchi_key",
        ],
    ]
    .drop_duplicates()
    .groupby(
        "inchi_connectivity_block",
        as_index=False,
    )
    .agg(
        n_full_inchi_keys_in_connectivity_block=(
            "resolved_full_inchi_key",
            "nunique",
        )
    )
)

perturbagen_structure_map = (
    perturbagen_structure_map
    .merge(
        full_inchi_by_block,
        on="inchi_connectivity_block",
        how="left",
        validate="many_to_one",
    )
)

perturbagen_structure_map[
    "n_full_inchi_keys_in_connectivity_block"
] = (
    perturbagen_structure_map[
        "n_full_inchi_keys_in_connectivity_block"
    ]
    .fillna(0)
    .astype(int)
)

# The first InChIKey block is retained only as a related-structure
# diagnostic; it never changes exact-structure membership.
perturbagen_structure_map[
    "shared_connectivity"
] = (
    perturbagen_structure_map[
        "n_full_inchi_keys_in_connectivity_block"
    ]
    .gt(1)
)


In [13]:
# =============================================================================
# Add exact-structure multiplicity diagnostics
# =============================================================================

exact_structure_sizes = (
    perturbagen_structure_map
    .groupby(
        "exact_structure_id",
        as_index=False,
    )
    .agg(
        n_pert_ids_exact_structure=(
            "pert_id",
            "nunique",
        )
    )
)

perturbagen_structure_map = (
    perturbagen_structure_map
    .merge(
        exact_structure_sizes,
        on="exact_structure_id",
        how="left",
        validate="many_to_one",
    )
)

# Shared exact structures expose redundant experimental pert_id
# representations; they are not independent pharmacological replications.
perturbagen_structure_map[
    "shared_exact_structure"
] = (
    perturbagen_structure_map[
        "n_pert_ids_exact_structure"
    ]
    .gt(1)
)


In [14]:
# =============================================================================
# Validate exact-structure reconciliation
# =============================================================================

if perturbagen_structure_map["pert_id"].duplicated().any():
    raise RuntimeError(
        "Perturbagen structure map is not one row per pert_id."
    )

if set(
    perturbagen_structure_map["pert_id"]
) != primary_perturbagen_ids:
    raise RuntimeError(
        "Perturbagen structure map does not preserve the frozen universe."
    )

resolved_keys_per_structure = (
    perturbagen_structure_map
    .groupby(
        "exact_structure_id"
    )[
        "resolved_full_inchi_key"
    ]
    .nunique()
)

if resolved_keys_per_structure.gt(1).any():
    raise RuntimeError(
        "One exact-structure group contains incompatible full InChIKeys."
    )

ambiguous_rows = (
    perturbagen_structure_map[
        "structure_identity_basis"
    ].eq(
        "SMILES_AMBIGUOUS_UNRESOLVED"
    )
)

if (
    perturbagen_structure_map.loc[
        ambiguous_rows,
        "n_pert_ids_exact_structure",
    ]
    .gt(1)
    .any()
):
    raise RuntimeError(
        "Ambiguous SMILES-only perturbagens were incorrectly collapsed."
    )

full_inchi_rows = (
    perturbagen_structure_map[
        "full_inchi_key"
    ].notna()
)

expected_full_ids = (
    "INCHIKEY::"
    + perturbagen_structure_map.loc[
        full_inchi_rows,
        "full_inchi_key",
    ].astype(str)
)

if not np.array_equal(
    perturbagen_structure_map.loc[
        full_inchi_rows,
        "exact_structure_id",
    ].to_numpy(),
    expected_full_ids.to_numpy(),
):
    raise RuntimeError(
        "Full InChIKey did not remain authoritative."
    )

print("EXACT-STRUCTURE RECONCILIATION: PASS")


EXACT-STRUCTURE RECONCILIATION: PASS


In [15]:
# =============================================================================
# Summarize chemical-identity outcomes
# =============================================================================

identity_summary = (
    perturbagen_structure_map[
        "structure_identity_basis"
    ]
    .value_counts()
    .rename_axis(
        "structure_identity_basis"
    )
    .reset_index(
        name="n_perturbagens"
    )
)

print(
    "Unique exact-structure groups:",
    f"{perturbagen_structure_map['exact_structure_id'].nunique():,}",
)
print(
    "Perturbagens sharing an exact structure:",
    f"{perturbagen_structure_map['shared_exact_structure'].sum():,}",
)
print(
    "Perturbagens in shared InChIKey connectivity blocks:",
    f"{perturbagen_structure_map['shared_connectivity'].sum():,}",
)

display(identity_summary)


Unique exact-structure groups: 7,877
Perturbagens sharing an exact structure: 73
Perturbagens in shared InChIKey connectivity blocks: 1,770


,structure_identity_basis,n_perturbagens
0,FULL_INCHIKEY,7389
1,UNKNOWN_PERT_ID,311
2,CANONICAL_SMILES_FALLBACK,214


## Structure-level descriptive consolidation

Notebook 701 significance remains perturbagen-level. Notebook 702 now summarizes every exact structure × program using all contributing pert_id values.

The structure-level connectivity value is the median resistance-like-oriented connectivity across contributing perturbagens. Minimum, maximum, and IQR are retained to expose within-structure heterogeneity. Direction concordance is descriptive only.

No best perturbagen representation is selected. No structure-level p-value, q-value, or rescue criterion is created.


In [16]:
# =============================================================================
# Join frozen primary inference to exact-structure identity
# =============================================================================

structure_primary = (
    primary_connectivity
    .merge(
        perturbagen_structure_map,
        on="pert_id",
        how="left",
        validate="many_to_one",
    )
)

if len(structure_primary) != len(
    primary_connectivity
):
    raise RuntimeError(
        "Structure join changed the frozen primary hypothesis count."
    )

if structure_primary[
    "exact_structure_id"
].isna().any():
    raise RuntimeError(
        "At least one frozen primary hypothesis lacks structure identity."
    )


In [17]:
# =============================================================================
# Define structure-summary helpers
# =============================================================================

def quantile_if_replicated(
    values,
    quantile,
):
    """Return a quantile only when >1 perturbagen represents the structure."""
    numeric = np.asarray(
        values,
        dtype=np.float64,
    )

    if len(numeric) < 2:
        return np.nan

    return float(
        np.quantile(
            numeric,
            quantile,
        )
    )


def summarize_direction(
    values,
):
    """Describe exact sign concordance without introducing a threshold."""
    numeric = np.asarray(
        values,
        dtype=np.float64,
    )

    if len(numeric) == 1:
        return "SINGLETON"

    negative = numeric < 0

    if negative.all():
        return "ALL_NEGATIVE"

    if (~negative).all():
        return "ALL_NONNEGATIVE"

    return "MIXED"


def ordered_status_json(
    values,
):
    """Serialize inherited Notebook 701 statuses in frozen display order."""
    statuses = sorted(
        set(
            values.astype(str)
        ),
        key=lambda status: (
            PRIMARY_STATUS_ORDER[
                status
            ]
        ),
    )

    return json.dumps(
        statuses,
        separators=(",", ":"),
    )


In [18]:
# =============================================================================
# Build structure-level connectivity and heterogeneity summary
# =============================================================================

structure_group_keys = [
    "program_id",
    "program_index",
    "exact_structure_id",
]

structure_connectivity_summary = (
    structure_primary
    .groupby(
        structure_group_keys,
        as_index=False,
        sort=True,
    )
    .agg(
        n_pert_ids=(
            "pert_id",
            "nunique",
        ),
        pert_ids_json=(
            "pert_id",
            json_unique_values,
        ),
        cmap_names_json=(
            "cmap_names_json",
            merge_json_value_lists,
        ),
        structure_identity_bases_json=(
            "structure_identity_basis",
            json_unique_values,
        ),
        resolved_full_inchi_key=(
            "resolved_full_inchi_key",
            first_nonmissing,
        ),
        canonical_smiles_json=(
            "canonical_smiles",
            json_unique_values,
        ),
        C_RL_median=(
            "C_RL",
            "median",
        ),
        C_RL_min=(
            "C_RL",
            "min",
        ),
        C_RL_max=(
            "C_RL",
            "max",
        ),
        C_RL_q25=(
            "C_RL",
            lambda values: quantile_if_replicated(
                values,
                0.25,
            ),
        ),
        C_RL_q75=(
            "C_RL",
            lambda values: quantile_if_replicated(
                values,
                0.75,
            ),
        ),
        fraction_negative_pert_ids=(
            "C_RL",
            lambda values: float(
                np.mean(
                    np.asarray(
                        values,
                        dtype=np.float64,
                    )
                    < 0
                )
            ),
        ),
        direction_concordance=(
            "C_RL",
            summarize_direction,
        ),
    )
)

structure_connectivity_summary[
    "C_RL_iqr"
] = (
    structure_connectivity_summary[
        "C_RL_q75"
    ]
    - structure_connectivity_summary[
        "C_RL_q25"
    ]
)


In [19]:
# =============================================================================
# Attach inherited primary-status composition to structure summaries
# =============================================================================

structure_status_summary = (
    structure_primary
    .groupby(
        structure_group_keys,
        as_index=False,
        sort=True,
    )
    .agg(
        primary_support_statuses_json=(
            "primary_support_status",
            ordered_status_json,
        ),
        n_primary_by_supported=(
            "primary_support_status",
            lambda values: int(
                values.eq(
                    "PRIMARY_BY_SUPPORTED"
                ).sum()
            ),
        ),
        n_inverse_not_by_supported=(
            "primary_support_status",
            lambda values: int(
                values.eq(
                    "INVERSE_NOT_BY_SUPPORTED"
                ).sum()
            ),
        ),
        n_non_inverse=(
            "primary_support_status",
            lambda values: int(
                values.eq(
                    "NON_INVERSE"
                ).sum()
            ),
        ),
        n_not_evaluable=(
            "primary_support_status",
            lambda values: int(
                values.eq(
                    "NOT_EVALUABLE"
                ).sum()
            ),
        ),
    )
)

structure_connectivity_summary = (
    structure_connectivity_summary
    .merge(
        structure_status_summary,
        on=structure_group_keys,
        how="left",
        validate="one_to_one",
    )
)


In [20]:
# =============================================================================
# Retain per-perturbagen primary status within each structure summary
# =============================================================================

structure_status_maps = []

for keys, block in structure_primary.groupby(
    structure_group_keys,
    sort=True,
):
    status_map = {
        str(pert_id): str(status)
        for pert_id, status in zip(
            block["pert_id"],
            block["primary_support_status"],
        )
    }

    structure_status_maps.append(
        {
            "program_id": keys[0],
            "program_index": int(
                keys[1]
            ),
            "exact_structure_id": keys[2],
            "pert_id_primary_status_json": (
                json.dumps(
                    status_map,
                    sort_keys=True,
                    separators=(",", ":"),
                )
            ),
        }
    )

structure_status_map = pd.DataFrame(
    structure_status_maps
)

structure_connectivity_summary = (
    structure_connectivity_summary
    .merge(
        structure_status_map,
        on=structure_group_keys,
        how="left",
        validate="one_to_one",
    )
)


In [21]:
# =============================================================================
# Validate structure-level descriptive consolidation
# =============================================================================

expected_structure_rows = (
    perturbagen_structure_map[
        "exact_structure_id"
    ].nunique()
    * len(PROGRAM_IDS)
)

if (
    len(structure_connectivity_summary)
    != expected_structure_rows
):
    raise RuntimeError(
        "Unexpected number of program × exact-structure summaries."
    )

if structure_connectivity_summary.duplicated(
    [
        "program_id",
        "exact_structure_id",
    ]
).any():
    raise RuntimeError(
        "Structure summary keys are not unique."
    )

if any(
    column.startswith(("p_", "q_"))
    for column in (
        structure_connectivity_summary.columns
    )
):
    raise RuntimeError(
        "Structure summary must not create inferential p/q fields."
    )

if any(
    "best" in column.casefold()
    for column in (
        structure_connectivity_summary.columns
    )
):
    raise RuntimeError(
        "Structure summary must not select a best perturbagen representation."
    )

print("STRUCTURE-LEVEL DESCRIPTIVE CONSOLIDATION: PASS")
print(
    "Program × structure rows:",
    f"{len(structure_connectivity_summary):,}",
)


STRUCTURE-LEVEL DESCRIPTIVE CONSOLIDATION: PASS
Program × structure rows: 23,631


## Prespecified sensitivity and diagnostic context

Notebook 701 already computed all inferential, sensitivity, and diagnostic quantities required here.

Notebook 702 does not recompute them and does not convert them into new significance gates. Weighted landmark, pooled-cell-line, is_hiq, and CMap-style scores are carried forward as descriptive sensitivity coordinates. The competitive pseudo-program quantity remains restricted to Notebook 701 primary-supported hypotheses and remains a descriptive tail fraction, not a new p-value.

Provider activity/reproducibility fields and the five frozen Hallmark scores remain contextual only. No diagnostic threshold is introduced.


In [22]:
# =============================================================================
# Validate sensitivity keys before catalog reshaping
# =============================================================================

base_sensitivity_types = set(
    SENSITIVITY_TYPES
)

observed_base_types = set(
    sensitivity_connectivity.loc[
        sensitivity_connectivity[
            "sensitivity_type"
        ].ne(
            "competitive_matched_pseudoprogram"
        ),
        "sensitivity_type",
    ]
    .unique()
)

if observed_base_types != base_sensitivity_types:
    raise RuntimeError(
        "Unexpected Notebook 701 base sensitivity types."
    )

for sensitivity_type in base_sensitivity_types:
    subset = sensitivity_connectivity.loc[
        sensitivity_connectivity[
            "sensitivity_type"
        ].eq(sensitivity_type)
    ]

    if len(subset) != PRIMARY_FAMILY_SIZE:
        raise RuntimeError(
            f"Unexpected row count for sensitivity: {sensitivity_type}"
        )

    if subset.duplicated(
        ["program_id", "pert_id"]
    ).any():
        raise RuntimeError(
            f"Duplicate sensitivity keys: {sensitivity_type}"
        )

print("NOTEBOOK 701 SENSITIVITY KEYS: VALIDATED")


NOTEBOOK 701 SENSITIVITY KEYS: VALIDATED


In [23]:
# =============================================================================
# Pivot base sensitivity scores into transparent catalog columns
# =============================================================================

sensitivity_context = (
    primary_connectivity[
        [
            "program_id",
            "pert_id",
        ]
    ]
    .copy()
)

for sensitivity_type, prefix in (
    SENSITIVITY_TYPES.items()
):
    subset = (
        sensitivity_connectivity.loc[
            sensitivity_connectivity[
                "sensitivity_type"
            ].eq(sensitivity_type),
            [
                "program_id",
                "pert_id",
                "sensitivity_score_RL",
                "sensitivity_evaluable",
            ],
        ]
        .rename(
            columns={
                "sensitivity_score_RL": (
                    f"{prefix}_score_RL"
                ),
                "sensitivity_evaluable": (
                    f"{prefix}_evaluable"
                ),
            }
        )
    )

    sensitivity_context = (
        sensitivity_context
        .merge(
            subset,
            on=[
                "program_id",
                "pert_id",
            ],
            how="left",
            validate="one_to_one",
        )
    )


In [24]:
# =============================================================================
# Attach competitive pseudo-program context
# =============================================================================

competitive_context = (
    sensitivity_connectivity.loc[
        sensitivity_connectivity[
            "sensitivity_type"
        ].eq(
            "competitive_matched_pseudoprogram"
        ),
        [
            "program_id",
            "pert_id",
            "competitive_draws",
            "competitive_g",
            "competitive_null_tail_fraction",
        ],
    ]
    .copy()
)

if competitive_context.duplicated(
    ["program_id", "pert_id"]
).any():
    raise RuntimeError(
        "Duplicate competitive-null sensitivity keys."
    )

supported_keys = set(
    map(
        tuple,
        primary_connectivity.loc[
            primary_connectivity[
                "primary_support_status"
            ].eq(
                "PRIMARY_BY_SUPPORTED"
            ),
            [
                "program_id",
                "pert_id",
            ],
        ]
        .to_numpy()
    )
)

competitive_keys = set(
    map(
        tuple,
        competitive_context[
            [
                "program_id",
                "pert_id",
            ]
        ]
        .to_numpy()
    )
)

if competitive_keys != supported_keys:
    raise RuntimeError(
        "Competitive-null rows do not exactly match the frozen "
        "PRIMARY_BY_SUPPORTED family."
    )

sensitivity_context = (
    sensitivity_context
    .merge(
        competitive_context,
        on=[
            "program_id",
            "pert_id",
        ],
        how="left",
        validate="one_to_one",
    )
)

sensitivity_context[
    "competitive_null_evaluated"
] = sensitivity_context[
    "competitive_null_tail_fraction"
].notna()


In [25]:
# =============================================================================
# Pivot perturbational-context diagnostics
# =============================================================================

if context_diagnostics.duplicated(
    [
        "pert_id",
        "diagnostic_name",
    ]
).any():
    raise RuntimeError(
        "Diagnostic perturbagen × name keys are not unique."
    )

observed_diagnostics = set(
    context_diagnostics[
        "diagnostic_name"
    ].unique()
)

if observed_diagnostics != set(
    DIAGNOSTIC_COLUMNS
):
    raise RuntimeError(
        "Notebook 701 diagnostic panel differs from the frozen set."
    )

diagnostic_context = (
    context_diagnostics
    .pivot(
        index="pert_id",
        columns="diagnostic_name",
        values="diagnostic_value",
    )
    .rename(
        columns=DIAGNOSTIC_COLUMNS
    )
    .reset_index()
)

diagnostic_context.columns.name = None

if len(diagnostic_context) != PRIMARY_PERTURBAGEN_COUNT:
    raise RuntimeError(
        "Diagnostic pivot does not retain every frozen perturbagen."
    )

print("SENSITIVITY / DIAGNOSTIC CONTEXT: PREPARED")


SENSITIVITY / DIAGNOSTIC CONTEXT: PREPARED


## Complete perturbational-hypothesis catalog

The catalog remains one row per frozen Notebook 701 program × pert_id hypothesis.

Primary status, p-values, q-values, connectivity, lineage breadth, and heterogeneity are inherited without modification. Exact-structure fields and the corresponding structure-level descriptive summary are appended for chemical-redundancy context. Prespecified sensitivities and diagnostics are appended as secondary/contextual columns.

Display order is deterministic: inherited primary support state first, then global BY q-value, resistance-like-oriented connectivity, program order, exact-structure ID, and pert_id. This ordering is for inspection only and does not define a new evidence tier or statistical threshold.


In [26]:
# =============================================================================
# Assemble the complete perturbational-hypothesis catalog
# =============================================================================

perturbational_hypothesis_catalog = (
    primary_connectivity
    .merge(
        perturbagen_structure_map,
        on="pert_id",
        how="left",
        validate="many_to_one",
    )
)

structure_catalog_columns = (
    structure_connectivity_summary[
        [
            "program_id",
            "exact_structure_id",
            "n_pert_ids",
            "C_RL_median",
            "C_RL_min",
            "C_RL_max",
            "C_RL_iqr",
            "fraction_negative_pert_ids",
            "direction_concordance",
            "primary_support_statuses_json",
            "pert_id_primary_status_json",
        ]
    ]
    .rename(
        columns={
            "n_pert_ids": (
                "structure_n_pert_ids"
            ),
            "C_RL_median": (
                "structure_C_RL_median"
            ),
            "C_RL_min": (
                "structure_C_RL_min"
            ),
            "C_RL_max": (
                "structure_C_RL_max"
            ),
            "C_RL_iqr": (
                "structure_C_RL_iqr"
            ),
            "fraction_negative_pert_ids": (
                "structure_fraction_negative_pert_ids"
            ),
            "direction_concordance": (
                "structure_direction_concordance"
            ),
            "primary_support_statuses_json": (
                "structure_primary_support_statuses_json"
            ),
            "pert_id_primary_status_json": (
                "structure_pert_id_primary_status_json"
            ),
        }
    )
)

perturbational_hypothesis_catalog = (
    perturbational_hypothesis_catalog
    .merge(
        structure_catalog_columns,
        on=[
            "program_id",
            "exact_structure_id",
        ],
        how="left",
        validate="many_to_one",
    )
)


In [27]:
# =============================================================================
# Attach sensitivity and diagnostic context to the complete catalog
# =============================================================================

perturbational_hypothesis_catalog = (
    perturbational_hypothesis_catalog
    .merge(
        sensitivity_context,
        on=[
            "program_id",
            "pert_id",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        diagnostic_context,
        on="pert_id",
        how="left",
        validate="many_to_one",
    )
)

if len(
    perturbational_hypothesis_catalog
) != PRIMARY_FAMILY_SIZE:
    raise RuntimeError(
        "Catalog assembly changed the frozen primary family size."
    )


In [28]:
# =============================================================================
# Apply deterministic display ordering without creating a new evidence tier
# =============================================================================

perturbational_hypothesis_catalog[
    "_primary_status_order"
] = perturbational_hypothesis_catalog[
    "primary_support_status"
].map(
    PRIMARY_STATUS_ORDER
)

if perturbational_hypothesis_catalog[
    "_primary_status_order"
].isna().any():
    raise RuntimeError(
        "Catalog contains an unknown inherited primary support state."
    )

perturbational_hypothesis_catalog = (
    perturbational_hypothesis_catalog
    .sort_values(
        [
            "_primary_status_order",
            "q_BY_global",
            "C_RL",
            "program_index",
            "exact_structure_id",
            "pert_id",
        ],
        kind="mergesort",
        na_position="last",
    )
    .drop(
        columns="_primary_status_order"
    )
    .reset_index(drop=True)
)

CATALOG_DISPLAY_ORDER = [
    "primary_support_status",
    "q_BY_global",
    "C_RL",
    "program_index",
    "exact_structure_id",
    "pert_id",
]


In [29]:
# =============================================================================
# Validate complete-catalog conservation of Notebook 701 inference
# =============================================================================

catalog_keys = (
    perturbational_hypothesis_catalog[
        [
            "program_id",
            "pert_id",
        ]
    ]
)

if catalog_keys.duplicated().any():
    raise RuntimeError(
        "Perturbational hypothesis catalog keys are not unique."
    )

primary_sorted = (
    primary_connectivity
    .sort_values(
        [
            "program_id",
            "pert_id",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

catalog_primary_sorted = (
    perturbational_hypothesis_catalog[
        primary_connectivity.columns
    ]
    .sort_values(
        [
            "program_id",
            "pert_id",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    catalog_primary_sorted,
    primary_sorted,
    check_dtype=False,
    check_exact=True,
)

primary_inference_exact = True

if any(
    column in perturbational_hypothesis_catalog.columns
    for column in (
        "target",
        "moa",
    )
):
    raise RuntimeError(
        "Target/MoA aggregation belongs to Notebook 703, not Notebook 702."
    )

print("PERTURBATIONAL HYPOTHESIS CATALOG: VALIDATED")
print(
    "Rows:",
    f"{len(perturbational_hypothesis_catalog):,}",
)
print(
    "PRIMARY_BY_SUPPORTED:",
    f"{perturbational_hypothesis_catalog['primary_support_status'].eq('PRIMARY_BY_SUPPORTED').sum():,}",
)


PERTURBATIONAL HYPOTHESIS CATALOG: VALIDATED
Rows: 23,742
PRIMARY_BY_SUPPORTED: 3,261


In [30]:
# =============================================================================
# Summarize Notebook 702 organization without defining new support tiers
# =============================================================================

catalog_status_summary = (
    perturbational_hypothesis_catalog[
        "primary_support_status"
    ]
    .value_counts()
    .reindex(
        PRIMARY_STATUS_ORDER
    )
    .rename_axis(
        "primary_support_status"
    )
    .reset_index(
        name="n_hypotheses"
    )
)

shared_structure_summary = (
    perturbagen_structure_map[
        [
            "shared_exact_structure",
            "shared_connectivity",
        ]
    ]
    .value_counts()
    .rename(
        "n_perturbagens"
    )
    .reset_index()
)

display(catalog_status_summary)
display(shared_structure_summary)


,primary_support_status,n_hypotheses
0,PRIMARY_BY_SUPPORTED,3261.0
1,INVERSE_NOT_BY_SUPPORTED,8605.0
2,NON_INVERSE,11876.0
3,NOT_EVALUABLE,NaN


,shared_exact_structure,shared_connectivity,n_perturbagens
0,False,False,6081
1,False,True,1760
2,True,False,63
3,True,True,10


## Persistence, provenance, and registry publication

Notebook 702 now writes only the four handoffs frozen by the Phase 7 contract.

The closure sequence follows current project policy:

persist outputs → validate outputs → finalize provenance → register artifacts → validate registry.

Processed outputs are immutable handoffs once registered. Re-running the notebook may reuse an identical frozen registry entry, but it must refuse to overwrite a non-identical frozen identity.


In [31]:
# =============================================================================
# Define Notebook 702 stable output paths
# =============================================================================

PHASE7_OUTPUT_DIR = (
    Paths.perturbational_hypotheses
)

PHASE7_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PERTURBAGEN_STRUCTURE_MAP_PATH = (
    PHASE7_OUTPUT_DIR
    / "702_perturbagen_structure_map.csv"
)

STRUCTURE_CONNECTIVITY_SUMMARY_PATH = (
    PHASE7_OUTPUT_DIR
    / "702_structure_connectivity_summary.csv"
)

PERTURBATIONAL_HYPOTHESIS_CATALOG_PATH = (
    PHASE7_OUTPUT_DIR
    / "702_perturbational_hypothesis_catalog.parquet"
)

ANALYSIS_METADATA_PATH = (
    PHASE7_OUTPUT_DIR
    / "702_analysis_metadata.json"
)


In [32]:
# =============================================================================
# Persist stable Notebook 702 tabular handoffs
# =============================================================================

perturbagen_structure_map = (
    perturbagen_structure_map
    .sort_values(
        "pert_id",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

structure_connectivity_summary = (
    structure_connectivity_summary
    .sort_values(
        [
            "program_index",
            "exact_structure_id",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

perturbagen_structure_map.to_csv(
    PERTURBAGEN_STRUCTURE_MAP_PATH,
    index=False,
)

structure_connectivity_summary.to_csv(
    STRUCTURE_CONNECTIVITY_SUMMARY_PATH,
    index=False,
)

perturbational_hypothesis_catalog.to_parquet(
    PERTURBATIONAL_HYPOTHESIS_CATALOG_PATH,
    index=False,
)

print("NOTEBOOK 702 TABULAR HANDOFFS: PERSISTED")
print(
    "Structure map rows:",
    f"{len(perturbagen_structure_map):,}",
)
print(
    "Structure summary rows:",
    f"{len(structure_connectivity_summary):,}",
)
print(
    "Hypothesis catalog rows:",
    f"{len(perturbational_hypothesis_catalog):,}",
)


NOTEBOOK 702 TABULAR HANDOFFS: PERSISTED
Structure map rows: 7,914
Structure summary rows: 23,631
Hypothesis catalog rows: 23,742


In [33]:
# =============================================================================
# Independently re-read Notebook 702 tabular handoffs
# =============================================================================

persisted_structure_map = pd.read_csv(
    PERTURBAGEN_STRUCTURE_MAP_PATH,
    low_memory=False,
)

persisted_structure_summary = pd.read_csv(
    STRUCTURE_CONNECTIVITY_SUMMARY_PATH,
    low_memory=False,
)

persisted_hypothesis_catalog = pd.read_parquet(
    PERTURBATIONAL_HYPOTHESIS_CATALOG_PATH
)

for label, persisted, expected in (
    (
        "perturbagen_structure_map",
        persisted_structure_map,
        perturbagen_structure_map,
    ),
    (
        "structure_connectivity_summary",
        persisted_structure_summary,
        structure_connectivity_summary,
    ),
    (
        "perturbational_hypothesis_catalog",
        persisted_hypothesis_catalog,
        perturbational_hypothesis_catalog,
    ),
):
    if persisted.shape != expected.shape:
        raise RuntimeError(
            f"Persisted shape mismatch: {label}"
        )

    if (
        persisted.columns.tolist()
        != expected.columns.tolist()
    ):
        raise RuntimeError(
            f"Persisted column mismatch: {label}"
        )

print("NOTEBOOK 702 TABULAR OUTPUT RE-READ: PASS")


NOTEBOOK 702 TABULAR OUTPUT RE-READ: PASS


In [34]:
# =============================================================================
# Prepare Notebook 702 provenance context
# =============================================================================

software_versions = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
}

try:
    software_versions[
        "pyarrow"
    ] = package_version(
        "pyarrow"
    )
except Exception:
    software_versions[
        "pyarrow"
    ] = None

identity_basis_counts = {
    str(key): int(value)
    for key, value in (
        perturbagen_structure_map[
            "structure_identity_basis"
        ]
        .value_counts()
        .items()
    )
}

primary_status_counts = {
    str(key): int(value)
    for key, value in (
        primary_connectivity[
            "primary_support_status"
        ]
        .value_counts()
        .items()
    )
}


In [35]:
# =============================================================================
# Define Notebook 702 chemical and structure metadata
# =============================================================================

chemical_identity_metadata = {
    "primary_experimental_unit": "pert_id",
    "cmap_name_role": (
        "descriptive metadata only"
    ),
    "literal_restricted_role": (
        "unavailable structure identity"
    ),
    "hierarchy": [
        "full InChIKey",
        "canonical SMILES fallback",
        "perturbagen-specific unresolved/unknown identity",
    ],
    "full_inchikey_authoritative": True,
    "smiles_only_attach_rule": (
        "attach to an existing full-InChIKey group only when "
        "the exact canonical SMILES maps to exactly one full "
        "InChIKey in the frozen primary annotation universe"
    ),
    "ambiguous_smiles_rule": (
        "retain as perturbagen-specific unresolved identity"
    ),
    "unknown_structure_rule": (
        "retain as perturbagen-specific unknown identity"
    ),
    "shared_connectivity_role": (
        "14-character InChIKey connectivity-block diagnostic only; "
        "does not collapse exact structures"
    ),
    "identity_basis_counts": (
        identity_basis_counts
    ),
    "exact_structure_groups": int(
        perturbagen_structure_map[
            "exact_structure_id"
        ].nunique()
    ),
}

structure_summary_metadata = {
    "unit": (
        "program_id × exact_structure_id"
    ),
    "connectivity": (
        "median of contributing pert_id C_RL values"
    ),
    "heterogeneity": [
        "C_RL min",
        "C_RL max",
        "C_RL IQR when >1 pert_id",
        "fraction negative pert_id values",
        "exact direction concordance",
    ],
    "primary_status_handling": (
        "all contributing Notebook 701 statuses retained; "
        "no structure-level significance test"
    ),
    "best_perturbagen_selection": False,
}


In [36]:
# =============================================================================
# Define Notebook 702 catalog and context metadata
# =============================================================================

catalog_metadata = {
    "unit": (
        "frozen Notebook 701 program_id × pert_id hypothesis"
    ),
    "complete_primary_family": True,
    "display_order": (
        CATALOG_DISPLAY_ORDER
    ),
    "display_order_role": (
        "deterministic inspection order only; no additional evidence tier"
    ),
    "opaque_composite_score": False,
}

sensitivity_metadata = {
    "source": (
        "phase7.701.sensitivity_connectivity"
    ),
    "base_types": list(
        SENSITIVITY_TYPES
    ),
    "competitive_null_role": (
        "descriptive tail fraction for Notebook 701 "
        "PRIMARY_BY_SUPPORTED hypotheses only"
    ),
    "can_rescue_primary_support": False,
}

diagnostic_metadata = {
    "source": (
        "phase7.701.perturbational_context_diagnostics"
    ),
    "names": list(
        DIAGNOSTIC_COLUMNS
    ),
    "role": (
        "contextual only; no exclusion, significance, or rescue gate"
    ),
}


In [37]:
# =============================================================================
# Build Notebook 702 analysis metadata
# =============================================================================

analysis_metadata = {
    "schema_version": 1,
    "notebook": "702_compound_prioritization",
    "analysis_role": (
        "phase7_exact_structure_reconciliation_"
        "chemical_redundancy_diagnostics_and_"
        "transparent_perturbational_hypothesis_organization"
    ),
    "specification_date": "2026-10-12",
    "contract": {
        "path": project_relative_path(
            contract_path
        ),
        "sha256": contract_sha256,
    },
    "input_artifact_hashes": input_hashes,
    "program_order": list(
        PROGRAM_IDS
    ),
    "primary_inference": {
        "source": (
            "phase7.701.primary_perturbagen_connectivity"
        ),
        "hypotheses": int(
            len(primary_connectivity)
        ),
        "perturbagens": int(
            primary_connectivity[
                "pert_id"
            ].nunique()
        ),
        "status_counts": (
            primary_status_counts
        ),
        "status_is_inherited_without_modification": True,
    },
    "chemical_identity": (
        chemical_identity_metadata
    ),
    "structure_summary": (
        structure_summary_metadata
    ),
    "catalog": (
        catalog_metadata
    ),
    "sensitivities": (
        sensitivity_metadata
    ),
    "diagnostics": (
        diagnostic_metadata
    ),
    "phase5_phase6_used": False,
    "target_moa_aggregation_performed": False,
    "software": software_versions,
    "stable_outputs": [
        "phase7.702.perturbagen_structure_map",
        "phase7.702.structure_connectivity_summary",
        "phase7.702.perturbational_hypothesis_catalog",
        "phase7.702.analysis_metadata",
    ],
    "interpretation_limitations": [
        (
            "Exact-structure consolidation is descriptive and does not "
            "replace perturbagen-level Notebook 701 inference."
        ),
        (
            "Multiple pert_id values sharing one exact structure are "
            "not independent pharmacological replications."
        ),
        (
            "The 14-character InChIKey connectivity block is a related-"
            "structure diagnostic and is not an exact-identity key."
        ),
        (
            "Sensitivity and perturbational-context fields remain "
            "secondary/contextual and cannot rescue primary support."
        ),
        (
            "Compound organization does not establish therapeutic "
            "reversal, efficacy, causal mechanism, or clinical relevance."
        ),
    ],
}


In [38]:
# =============================================================================
# Persist Notebook 702 analysis metadata
# =============================================================================

with ANALYSIS_METADATA_PATH.open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        analysis_metadata,
        handle,
        indent=2,
        sort_keys=True,
    )
    handle.write("\n")

print("NOTEBOOK 702 ANALYSIS METADATA: PERSISTED")


NOTEBOOK 702 ANALYSIS METADATA: PERSISTED


In [39]:
# =============================================================================
# Define Notebook 702 stable artifact outputs
# =============================================================================

artifact_outputs = {
    "phase7.702.perturbagen_structure_map": {
        "path": PERTURBAGEN_STRUCTURE_MAP_PATH,
        "format": "csv",
        "artifact_role": "handoff",
    },
    "phase7.702.structure_connectivity_summary": {
        "path": STRUCTURE_CONNECTIVITY_SUMMARY_PATH,
        "format": "csv",
        "artifact_role": "handoff",
    },
    "phase7.702.perturbational_hypothesis_catalog": {
        "path": PERTURBATIONAL_HYPOTHESIS_CATALOG_PATH,
        "format": "parquet",
        "artifact_role": "handoff",
    },
    "phase7.702.analysis_metadata": {
        "path": ANALYSIS_METADATA_PATH,
        "format": "json",
        "artifact_role": "metadata",
    },
}


In [40]:
# =============================================================================
# Fingerprint independently re-read Notebook 702 outputs
# =============================================================================

artifact_identity_rows = []

for artifact_id, spec in (
    artifact_outputs.items()
):
    artifact_path = spec["path"]

    if (
        not artifact_path.is_file()
        or artifact_path.stat().st_size == 0
    ):
        raise RuntimeError(
            f"Missing or empty stable output: {artifact_id}"
        )

    if spec["format"] == "csv":
        persisted = pd.read_csv(
            artifact_path,
            low_memory=False,
        )
        shape = [
            int(persisted.shape[0]),
            int(persisted.shape[1]),
        ]

    elif spec["format"] == "parquet":
        persisted = pd.read_parquet(
            artifact_path
        )
        shape = [
            int(persisted.shape[0]),
            int(persisted.shape[1]),
        ]

    elif spec["format"] == "json":
        with artifact_path.open(
            "r",
            encoding="utf-8",
        ) as handle:
            persisted = json.load(handle)

        if persisted != analysis_metadata:
            raise RuntimeError(
                "Persisted analysis metadata differs from memory."
            )

        shape = None

    else:
        raise ValueError(
            f"Unknown output format: {spec['format']}"
        )

    artifact_identity_rows.append(
        {
            "artifact_id": artifact_id,
            "path": project_relative_path(
                artifact_path
            ),
            "artifact_role": spec[
                "artifact_role"
            ],
            "shape": shape,
            "size_bytes": int(
                artifact_path.stat().st_size
            ),
            "sha256": calculate_sha256(
                artifact_path
            ),
        }
    )

artifact_identities = pd.DataFrame(
    artifact_identity_rows
)

print("NOTEBOOK 702 OUTPUT FINGERPRINTING: PASS")
display(artifact_identities)


NOTEBOOK 702 OUTPUT FINGERPRINTING: PASS


,artifact_id,path,artifact_role,shape,size_bytes,sha256
0,phase7.702.perturbagen_structure_map,data/processed/perturbational_hypotheses/702_p...,handoff,"[7914, 15]",1885376,66377c8b37c2fd09bdd9de3edc9045153af65c7882770d...
1,phase7.702.structure_connectivity_summary,data/processed/perturbational_hypotheses/702_s...,handoff,"[23631, 23]",8684641,ef41354bd158f7613535c5e8ebec73abe2b965ef9047db...
2,phase7.702.perturbational_hypothesis_catalog,data/processed/perturbational_hypotheses/702_p...,handoff,"[23742, 72]",5224473,a17595f9cc634838da6a367012293bfc0576a2a99286df...
3,phase7.702.analysis_metadata,data/processed/perturbational_hypotheses/702_a...,metadata,None,5311,6c06134cb6d57a4ec5ae015b62faa78e380e5acec4433b...


In [41]:
# =============================================================================
# Define Notebook 702 registry dependencies
# =============================================================================

NOTEBOOK_702_PATH = (
    "notebooks/phase7_perturbational_hypotheses/"
    "702_compound_prioritization.ipynb"
)

artifact_inputs = {
    "phase7.702.perturbagen_structure_map": [
        {
            "type": "artifact",
            "artifact_id": (
                "phase1.110.primary_compound_annotation"
            ),
        },
        {
            "type": "artifact",
            "artifact_id": (
                "phase7.701.primary_perturbagen_connectivity"
            ),
        },
    ],
    "phase7.702.structure_connectivity_summary": [
        {
            "type": "artifact",
            "artifact_id": (
                "phase7.702.perturbagen_structure_map"
            ),
        },
        {
            "type": "artifact",
            "artifact_id": (
                "phase7.701.primary_perturbagen_connectivity"
            ),
        },
    ],
    "phase7.702.perturbational_hypothesis_catalog": [
        {
            "type": "artifact",
            "artifact_id": (
                "phase7.701.primary_perturbagen_connectivity"
            ),
        },
        {
            "type": "artifact",
            "artifact_id": (
                "phase7.701.sensitivity_connectivity"
            ),
        },
        {
            "type": "artifact",
            "artifact_id": (
                "phase7.701.perturbational_context_diagnostics"
            ),
        },
        {
            "type": "artifact",
            "artifact_id": (
                "phase7.702.perturbagen_structure_map"
            ),
        },
        {
            "type": "artifact",
            "artifact_id": (
                "phase7.702.structure_connectivity_summary"
            ),
        },
    ],
    "phase7.702.analysis_metadata": [
        {
            "type": "artifact",
            "artifact_id": (
                "phase7.701.analysis_metadata"
            ),
        },
        {
            "type": "artifact",
            "artifact_id": (
                "phase7.702.perturbagen_structure_map"
            ),
        },
        {
            "type": "artifact",
            "artifact_id": (
                "phase7.702.structure_connectivity_summary"
            ),
        },
        {
            "type": "artifact",
            "artifact_id": (
                "phase7.702.perturbational_hypothesis_catalog"
            ),
        },
    ],
}


In [42]:
# =============================================================================
# Register frozen Notebook 702 artifacts
# =============================================================================

identity_index = (
    artifact_identities
    .set_index("artifact_id")
)

registry_entries = {}

for artifact_id in artifact_outputs:
    identity = identity_index.loc[
        artifact_id
    ]

    registry_entries[
        artifact_id
    ] = {
        "path": identity["path"],
        "phase": 7,
        "status": "frozen",
        "artifact_role": identity[
            "artifact_role"
        ],
        "producer": {
            "type": "notebook",
            "path": NOTEBOOK_702_PATH,
        },
        "shape": (
            identity["shape"]
            if isinstance(
                identity["shape"],
                list,
            )
            else None
        ),
        "size_bytes": int(
            identity["size_bytes"]
        ),
        "sha256": identity["sha256"],
        "inputs": artifact_inputs[
            artifact_id
        ],
    }

updated_registry = load_artifact_registry()

for artifact_id, entry in (
    registry_entries.items()
):
    existing = updated_registry[
        "artifacts"
    ].get(artifact_id)

    if (
        existing is not None
        and existing != entry
    ):
        raise RuntimeError(
            "A non-identical frozen Notebook 702 registry "
            f"entry already exists: {artifact_id}"
        )

    updated_registry[
        "artifacts"
    ][artifact_id] = entry

validate_artifact_registry(
    updated_registry
)

Paths.artifact_registry.write_text(
    json.dumps(
        updated_registry,
        indent=2,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)

print("NOTEBOOK 702 ARTIFACT REGISTRATION: PASS")
print(
    "Registered artifacts:",
    len(registry_entries),
)


NOTEBOOK 702 ARTIFACT REGISTRATION: PASS
Registered artifacts: 4


In [43]:
# =============================================================================
# Prepare final Notebook 702 closure QA
# =============================================================================

final_registry = load_artifact_registry()

registered_702 = {
    artifact_id: artifact
    for artifact_id, artifact in (
        final_registry[
            "artifacts"
        ].items()
    )
    if artifact_id.startswith(
        "phase7.702."
    )
}

catalog_primary_check = (
    perturbational_hypothesis_catalog[
        primary_connectivity.columns
    ]
    .sort_values(
        [
            "program_id",
            "pert_id",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


In [44]:
# =============================================================================
# Evaluate Notebook 702 chemical and structure closure conditions
# =============================================================================

qa_results = {
    "four_artifacts_registered": (
        set(registered_702)
        == set(artifact_outputs)
    ),
    "structure_map_7914": (
        len(perturbagen_structure_map)
        == PRIMARY_PERTURBAGEN_COUNT
    ),
    "structure_map_keys_unique": (
        not perturbagen_structure_map[
            "pert_id"
        ].duplicated().any()
    ),
    "structure_map_complete_universe": (
        set(
            perturbagen_structure_map[
                "pert_id"
            ]
        )
        == primary_perturbagen_ids
    ),
    "full_inchikey_authoritative": (
        (
            perturbagen_structure_map.loc[
                perturbagen_structure_map[
                    "full_inchi_key"
                ].notna(),
                "structure_identity_basis",
            ]
            == "FULL_INCHIKEY"
        ).all()
    ),
    "ambiguous_smiles_not_collapsed": (
        perturbagen_structure_map.loc[
            perturbagen_structure_map[
                "structure_identity_basis"
            ].eq(
                "SMILES_AMBIGUOUS_UNRESOLVED"
            ),
            "n_pert_ids_exact_structure",
        ]
        .le(1)
        .all()
    ),
    "no_incompatible_inchikey_merge": (
        resolved_keys_per_structure
        .le(1)
        .all()
    ),
    "structure_summary_complete": (
        len(
            structure_connectivity_summary
        )
        == expected_structure_rows
    ),
    "structure_summary_keys_unique": (
        not structure_connectivity_summary
        .duplicated(
            [
                "program_id",
                "exact_structure_id",
            ]
        )
        .any()
    ),
    "structure_summary_no_inferential_pq": (
        not any(
            column.startswith(
                ("p_", "q_")
            )
            for column in (
                structure_connectivity_summary.columns
            )
        )
    ),
}


In [45]:
# =============================================================================
# Evaluate Notebook 702 catalog and analysis-boundary conditions
# =============================================================================

qa_results.update(
    {
        "catalog_23742": (
            len(
                perturbational_hypothesis_catalog
            )
            == PRIMARY_FAMILY_SIZE
        ),
        "catalog_keys_unique": (
            not perturbational_hypothesis_catalog
            .duplicated(
                [
                    "program_id",
                    "pert_id",
                ]
            )
            .any()
        ),
        "catalog_primary_inference_exact": (
            primary_inference_exact
        ),
        "competitive_context_primary_supported_only": (
            competitive_keys
            == supported_keys
        ),
        "target_moa_deferred_to_703": (
            "target"
            not in perturbational_hypothesis_catalog.columns
            and "moa"
            not in perturbational_hypothesis_catalog.columns
        ),
        "no_phase5_phase6_inputs": (
            not any(
                artifact_id.startswith(
                    ("phase5.", "phase6.")
                )
                for artifact_id in (
                    INPUT_ARTIFACT_IDS
                )
            )
        ),
    }
)


In [46]:
# =============================================================================
# Validate registered Notebook 702 output identities
# =============================================================================

for artifact_id, spec in (
    artifact_outputs.items()
):
    entry = registered_702[
        artifact_id
    ]

    qa_results[
        f"{artifact_id}:path"
    ] = (
        (
            Paths.root
            / entry["path"]
        ).resolve()
        == spec["path"].resolve()
    )

    qa_results[
        f"{artifact_id}:sha256"
    ] = (
        calculate_sha256(
            spec["path"]
        )
        == entry["sha256"]
    )


In [47]:
# =============================================================================
# Report final Notebook 702 closure status
# =============================================================================

validate_artifact_registry(
    final_registry
)

qa_series = pd.Series(
    qa_results,
    name="PASS",
)

display(qa_series)

if not qa_series.all():
    failed = qa_series.loc[
        ~qa_series
    ].index.tolist()

    raise RuntimeError(
        "Notebook 702 closure QA failed: "
        + ", ".join(failed)
    )

print("NOTEBOOK 702 FINAL QA: PASS")
print(
    "Primary hypotheses preserved:",
    f"{len(perturbational_hypothesis_catalog):,}",
)
print(
    "Exact-structure groups:",
    f"{perturbagen_structure_map['exact_structure_id'].nunique():,}",
)
print(
    "PRIMARY_BY_SUPPORTED preserved:",
    f"{perturbational_hypothesis_catalog['primary_support_status'].eq('PRIMARY_BY_SUPPORTED').sum():,}",
)
print(
    "Stable outputs registered:",
    "4/4",
)


four_artifacts_registered                              True
structure_map_7914                                     True
structure_map_keys_unique                              True
structure_map_complete_universe                        True
full_inchikey_authoritative                            True
ambiguous_smiles_not_collapsed                         True
no_incompatible_inchikey_merge                         True
structure_summary_complete                             True
structure_summary_keys_unique                          True
structure_summary_no_inferential_pq                    True
catalog_23742                                          True
catalog_keys_unique                                    True
catalog_primary_inference_exact                        True
competitive_context_primary_supported_only             True
target_moa_deferred_to_703                             True
no_phase5_phase6_inputs                                True
phase7.702.perturbagen_structure_map:pat

NOTEBOOK 702 FINAL QA: PASS
Primary hypotheses preserved: 23,742
Exact-structure groups: 7,877
PRIMARY_BY_SUPPORTED preserved: 3,261
Stable outputs registered: 4/4


## Notebook completion

Notebook 702 is complete only when the final QA cell passes.

The Notebook 701 perturbagen-level inferential family and primary support states remain unchanged. Chemical reconciliation exposes exact-structure redundancy and within-structure heterogeneity without choosing favorable perturbagen representations or creating structure-level significance.

The complete perturbational-hypothesis catalog carries forward primary connectivity, lineage-aware context, prespecified sensitivities, and perturbational-context diagnostics for transparent downstream use.

Target and mechanism-of-action aggregation remains reserved for Notebook 703. Final cross-evidence therapeutic prioritization remains reserved for Phase 9. The outputs of this notebook are computational perturbational hypotheses and chemical-organization handoffs, not therapeutic recommendations or validated mechanisms.
